In [1]:
# ═══════════════════════════════════════════════════
# CELL 1 — Setup
# ═══════════════════════════════════════════════════
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
from sklearn.model_selection import cross_val_score
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

DATA_PATH = "../data/processed/"

X_train = pd.read_csv(f"{DATA_PATH}X_train.csv")
X_test = pd.read_csv(f"{DATA_PATH}X_test.csv")
y_train = pd.read_csv(f"{DATA_PATH}y_train.csv").values.ravel()
y_test = pd.read_csv(f"{DATA_PATH}y_test.csv").values.ravel()

print(f"X_train: {X_train.shape}")
print(f"X_test:  {X_test.shape}")
print("✅ Loaded")

X_train: (3352090, 48)
X_test:  (504160, 48)
✅ Loaded


In [2]:
# ═══════════════════════════════════════════════════
# CHECK 1 — Near-Duplicate Detection
# Are there rows in X_test that are nearly identical
# to rows in X_train?
# ═══════════════════════════════════════════════════
from sklearn.neighbors import NearestNeighbors

print("=" * 60)
print("CHECK 1: NEAR-DUPLICATE DETECTION")
print("=" * 60)

# Sample for speed (full dataset is too large)
sample_size = 10000
np.random.seed(42)

test_sample_idx = np.random.choice(len(X_test), sample_size, replace=False)
X_test_sample = X_test.iloc[test_sample_idx]

# Find nearest neighbor in training set for each test sample
nn = NearestNeighbors(n_neighbors=1, algorithm='auto', n_jobs=-1)
nn.fit(X_train)

distances, indices = nn.kneighbors(X_test_sample)

print(f"\nNearest neighbor distances (test → train):")
print(f"  Min distance:    {distances.min():.6f}")
print(f"  Mean distance:   {distances.mean():.6f}")
print(f"  Median distance: {np.median(distances):.6f}")
print(f"  Max distance:    {distances.max():.6f}")
print(f"\n  Exact matches (distance=0): {(distances == 0).sum()}")
print(f"  Near matches (distance<0.01): {(distances < 0.01).sum()}")
print(f"  Near matches (distance<0.1): {(distances < 0.1).sum()}")

pct_near = (distances < 0.01).sum() / sample_size * 100
print(f"\n  ⚠️ {pct_near:.1f}% of test samples have a near-duplicate in training")

if pct_near > 50:
    print("  🔴 HIGH LEAKAGE RISK — Many test samples are nearly identical to training")
elif pct_near > 20:
    print("  🟡 MODERATE RISK — Some overlap exists")
else:
    print("  🟢 LOW RISK — Test set appears sufficiently different")

CHECK 1: NEAR-DUPLICATE DETECTION

Nearest neighbor distances (test → train):
  Min distance:    0.000000
  Mean distance:   0.045413
  Median distance: 0.000802
  Max distance:    6.883553

  Exact matches (distance=0): 107
  Near matches (distance<0.01): 7316
  Near matches (distance<0.1): 9011

  ⚠️ 73.2% of test samples have a near-duplicate in training
  🔴 HIGH LEAKAGE RISK — Many test samples are nearly identical to training


In [3]:
# ═══════════════════════════════════════════════════
# CHECK 2 — Cross-Validation on Original Data
# If CV scores are also ~1.0, the dataset is genuinely easy.
# If CV scores are much lower, there's leakage.
# ═══════════════════════════════════════════════════
print("\n" + "=" * 60)
print("CHECK 2: CROSS-VALIDATION (NO SMOTE)")
print("=" * 60)

# Load original processed data (before SMOTE)
df = pd.read_csv(f"{DATA_PATH}processed_traffic.csv", low_memory=False)

exclude_cols = ['label', 'label_binary', 'label_multi']
non_numeric = df.select_dtypes(exclude=[np.number]).columns.tolist()
drop_cols = list(set(exclude_cols + non_numeric))
drop_cols = [col for col in drop_cols if col in df.columns]

X_original = df.drop(columns=drop_cols)
y_original = df['label_binary'].values

# Use a smaller sample for speed
sample_idx = np.random.choice(len(X_original), min(100000, len(X_original)), replace=False)
X_sample = X_original.iloc[sample_idx]
y_sample = y_original[sample_idx]

print(f"  Running 5-fold CV on {len(X_sample):,} samples...")
print(f"  (No SMOTE, no scaling — raw features)")

# XGBoost CV
xgb_cv = XGBClassifier(
    n_estimators=100, max_depth=6,
    random_state=42, n_jobs=-1,
    eval_metric='logloss', use_label_encoder=False
)
xgb_scores = cross_val_score(xgb_cv, X_sample, y_sample,
                              cv=5, scoring='roc_auc', n_jobs=-1)

print(f"\n  XGBoost 5-Fold CV AUC:")
print(f"    Fold scores: {xgb_scores.round(4)}")
print(f"    Mean AUC:    {xgb_scores.mean():.4f} ± {xgb_scores.std():.4f}")

if xgb_scores.mean() > 0.99:
    print("  🟢 CV also shows ~1.0 — Dataset is genuinely separable")
elif xgb_scores.mean() > 0.95:
    print("  🟡 CV shows high but not perfect — Some leakage possible")
else:
    print("  🔴 CV much lower than test AUC — Likely data leakage")


CHECK 2: CROSS-VALIDATION (NO SMOTE)
  Running 5-fold CV on 100,000 samples...
  (No SMOTE, no scaling — raw features)


/Users/apple/NetSentinel/venv_tf/lib/python3.11/site-packages/xgboost/training.py:200: UserWarning: [23:39:41] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/Users/apple/NetSentinel/venv_tf/lib/python3.11/site-packages/xgboost/training.py:200: UserWarning: [23:39:41] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/Users/apple/NetSentinel/venv_tf/lib/python3.11/site-packages/xgboost/training.py:200: UserWarning: [23:39:41] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/Users/apple/NetSentinel/venv_tf/lib/python3.11/site-packages/xgboost/training.py:200: UserWarning: [23:39:41] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:782: 
Parameters:


  XGBoost 5-Fold CV AUC:
    Fold scores: [0.9996 0.9999 1.     1.     0.9998]
    Mean AUC:    0.9999 ± 0.0001
  🟢 CV also shows ~1.0 — Dataset is genuinely separable


In [4]:
# ═══════════════════════════════════════════════════
# CHECK 3 — Train on Monday, Test on Friday
# Completely separate time periods = no leakage possible
# ═══════════════════════════════════════════════════
print("\n" + "=" * 60)
print("CHECK 3: TEMPORAL SPLIT (Monday train → Friday test)")
print("=" * 60)

import os

raw_path = "../data/raw/"
files = os.listdir(raw_path)

# Find Monday and Friday files
monday_files = [f for f in files if 'Monday' in f]
friday_files = [f for f in files if 'Friday' in f]

if monday_files and friday_files:
    # Load Monday (all benign)
    df_monday = pd.read_csv(os.path.join(raw_path, monday_files[0]), low_memory=False)
    df_monday.columns = df_monday.columns.str.strip().str.replace(' ', '_').str.replace('/', '_').str.lower()

    # Load Friday (DDoS + PortScan + Benign)
    dfs_friday = []
    for f in friday_files:
        temp = pd.read_csv(os.path.join(raw_path, f), low_memory=False)
        temp.columns = temp.columns.str.strip().str.replace(' ', '_').str.replace('/', '_').str.lower()
        dfs_friday.append(temp)
    df_friday = pd.concat(dfs_friday, ignore_index=True)

    # Create binary labels
    df_monday['label_binary'] = (df_monday['label'] != 'BENIGN').astype(int)
    df_friday['label_binary'] = (df_friday['label'] != 'BENIGN').astype(int)

    # Get common numeric columns
    exclude = ['label', 'label_binary']
    mon_numeric = df_monday.select_dtypes(include=[np.number]).columns.tolist()
    fri_numeric = df_friday.select_dtypes(include=[np.number]).columns.tolist()
    common_cols = [c for c in mon_numeric if c in fri_numeric and c not in exclude]

    # Clean
    df_monday = df_monday[common_cols + ['label_binary']].replace([np.inf, -np.inf], np.nan).dropna()
    df_friday = df_friday[common_cols + ['label_binary']].replace([np.inf, -np.inf], np.nan).dropna()

    # Sample for speed
    mon_sample = df_monday.sample(min(50000, len(df_monday)), random_state=42)
    fri_sample = df_friday.sample(min(50000, len(df_friday)), random_state=42)

    # Combine Monday + some Friday benign for training
    # Test on Friday attacks
    X_temporal_train = mon_sample[common_cols]
    y_temporal_train = mon_sample['label_binary'].values

    X_temporal_test = fri_sample[common_cols]
    y_temporal_test = fri_sample['label_binary'].values

    print(f"  Train (Monday): {len(X_temporal_train):,} samples "
          f"({y_temporal_train.sum()} attacks)")
    print(f"  Test (Friday):  {len(X_temporal_test):,} samples "
          f"({y_temporal_test.sum()} attacks)")

    # Train XGBoost
    from sklearn.metrics import roc_auc_score
    from sklearn.preprocessing import StandardScaler

    scaler = StandardScaler()
    X_tr = scaler.fit_transform(X_temporal_train)
    X_te = scaler.transform(X_temporal_test)

    xgb_temporal = XGBClassifier(
        n_estimators=100, max_depth=6,
        random_state=42, n_jobs=-1,
        eval_metric='logloss', use_label_encoder=False
    )
    xgb_temporal.fit(X_tr, y_temporal_train)
    y_scores = xgb_temporal.predict_proba(X_te)[:, 1]
    temporal_auc = roc_auc_score(y_temporal_test, y_scores)

    print(f"\n  Temporal Split AUC: {temporal_auc:.4f}")

    if temporal_auc > 0.99:
        print("  🟢 Still near-perfect — Dataset is genuinely easy")
    elif temporal_auc > 0.90:
        print("  🟡 Good but lower — Some leakage in original split")
    else:
        print("  🔴 Much lower — Original results likely inflated by leakage")
else:
    print("  ⚠️ Raw files not found. Skip this check.")


CHECK 3: TEMPORAL SPLIT (Monday train → Friday test)
  Train (Monday): 50,000 samples (0 attacks)
  Test (Friday):  50,000 samples (20566 attacks)


/Users/apple/NetSentinel/venv_tf/lib/python3.11/site-packages/xgboost/training.py:200: UserWarning: [23:40:00] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)



  Temporal Split AUC: 0.5000
  🔴 Much lower — Original results likely inflated by leakage


In [5]:
# ═══════════════════════════════════════════════════
# CHECK 4 — Summary & Verdict
# ═══════════════════════════════════════════════════
print("\n" + "═" * 60)
print("  LEAKAGE DIAGNOSIS SUMMARY")
print("═" * 60)

print("""
  FACTORS CONTRIBUTING TO HIGH AUC:

  1. CIC-IDS2017 is a well-known "easy" dataset for tree models.
     Published papers routinely report 99%+ accuracy.

  2. Attack traffic has very distinct patterns:
     - DDoS: massive packet floods → obvious in features
     - PortScan: rapid sequential connections → obvious timing
     - Brute Force: repeated auth attempts → obvious repetition

  3. SMOTE may inflate training set with samples close to test.

  4. Near-duplicate flows from same attack sessions may exist
     in both train and test sets.

  VERDICT:
  ────────
  AUC = 1.0 on CIC-IDS2017 is COMMON in literature.
  It doesn't necessarily mean your pipeline is wrong.
  BUT it means your model's real-world performance would
  likely be lower on truly unseen traffic.

  RECOMMENDATION:
  ────────────────
  • Report the result honestly
  • Acknowledge dataset limitations
  • Run temporal split to show more realistic performance
  • Mention that real deployment would need continuous retraining
""")


════════════════════════════════════════════════════════════
  LEAKAGE DIAGNOSIS SUMMARY
════════════════════════════════════════════════════════════

  FACTORS CONTRIBUTING TO HIGH AUC:

  1. CIC-IDS2017 is a well-known "easy" dataset for tree models.
     Published papers routinely report 99%+ accuracy.

  2. Attack traffic has very distinct patterns:
     - DDoS: massive packet floods → obvious in features
     - PortScan: rapid sequential connections → obvious timing
     - Brute Force: repeated auth attempts → obvious repetition

  3. SMOTE may inflate training set with samples close to test.

  4. Near-duplicate flows from same attack sessions may exist
     in both train and test sets.

  VERDICT:
  ────────
  AUC = 1.0 on CIC-IDS2017 is COMMON in literature.
  It doesn't necessarily mean your pipeline is wrong.
  BUT it means your model's real-world performance would
  likely be lower on truly unseen traffic.

  RECOMMENDATION:
  ────────────────
  • Report the result honestly